In [ ]:
#maintenance by LL
#on 03/12/2021

from bs4 import BeautifulSoup
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
from time import sleep
import datetime
from pandas import ExcelWriter
import os

print("Running MY CBMAL Web Scraping Tool v.1.0")
now=datetime.datetime.now()
scriptfolder=os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

filename= 'MY CBMAL SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

regdict={'MY CBMAL 1': ['https://www.bnm.gov.my/web/guest/commercial-banks'], 
		 'MY CBMAL 2': ['https://www.bnm.gov.my/web/guest/islamic-banks'], 
		 'MY CBMAL 3': ['https://www.bnm.gov.my/web/guest/international-islamic-banks'], 
		 'MY CBMAL 4': ['https://www.bnm.gov.my/web/guest/investment-banks'], 
		 'MY CBMAL 5': ['https://www.bnm.gov.my/web/guest/other-financial-institutions'], 
		 'MY CBMAL 6': ['https://www.bnm.gov.my/web/guest/life-business',
						'https://www.bnm.gov.my/web/guest/general-business',
						'https://www.bnm.gov.my/web/guest/life-and-general-reinsurance-business',
						'https://www.bnm.gov.my/web/guest/life-reinsurance-business',
						'https://www.bnm.gov.my/web/guest/general-reinsurance-business',
						'https://www.bnm.gov.my/web/guest/takaful-operators',
						'https://www.bnm.gov.my/web/guest/retakaful-operators'], 
		 'MY CBMAL 7': ['https://www.bnm.gov.my/-/list-of-approved-money-brokers'], 
		 'MY CBMAL 8': ['https://www.bnm.gov.my/-/approved-insurance-brokers'], 
		 'MY CBMAL 9': ['https://www.bnm.gov.my/-/approved-insurance-and-takaful-brokers'], 
		 'MY CBMAL 10': ['https://www.bnm.gov.my/-/approved-takaful-brokers-specialised-'], 
		 'MY CBMAL 11': ['https://www.bnm.gov.my/-/registered-adjusters'], 
		 'MY CBMAL 12': ['https://www.bnm.gov.my/-/approved-financial-advisers'],
		 'MY CBMAL 13': ['https://www.bnm.gov.my/-/approved-international-marine-aviation-and-transit-mat-insurance-brokers'],
		 'MY CBMAL 14': ['https://www.bnm.gov.my/-/approved-islamic-financial-advisers'],
		 'MY CBMAL 15': ['https://www.bnm.gov.my/-/approved-electronic-trading-platforms-etp-'],
		 'MY CBMAL 16': ['https://www.bnm.gov.my/list-of-financial-holding-companies']}
		 
driver = webdriver.Chrome()
driver.maximize_window()

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')

for reg in regdict:
    print('Working with {}.'.format(reg))
    urls = regdict[reg]
    for url in urls:
        clinks=[]
        driver.get(url)
        sleep(2)
        soup = BeautifulSoup(driver.page_source.replace('<br>','***').replace('<br/>','***'), 'html.parser')
        typology = soup.find('span', {'class': 'active breadcrumb-text-truncate'}).text.strip()
        if soup.find('table', {'class': 'Press-table'}) is not None:
            table = soup.find('table', {'class': 'Press-table'})
            entities = table.find_all('a', href=True)
            for entity in entities:
                driver.get(entity['href'])
                sleep(0.25)
                sqldict['Name'].append(entity.text.strip())
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                labels = soup.find_all('td', {'class':'txt_stdContentFont'})
                for label in labels:
                    if 'HQ Address' in label.text and len(sqldict['Name'])>len(sqldict['Address_1']):
                        value = label.parent.find_all('td')[1].text.strip()
                        sqldict['Address_1'].append(value)
                    elif 'Telephone' in label.text and len(sqldict['Name'])>len(sqldict['Phone']):
                        value = label.parent.find_all('td')[1].text.strip()
                        sqldict['Phone'].append(value)
                    elif 'Website' in label.text and len(sqldict['Name'])>len(sqldict['Website']):
                        value = label.parent.find_all('td')[1].text.strip()
                        sqldict['Website'].append(value)
                    elif 'Facsimile' in label.text and len(sqldict['Name'])>len(sqldict['Fax']):
                        value = label.parent.find_all('td')[1].text.strip()
                        sqldict['Fax'].append(value)
                    elif 'E-mail' in label.text and len(sqldict['Name'])>len(sqldict['Email']):
                        value = label.parent.find_all('td')[1].text.strip()
                        sqldict['Email'].append(value)
                sqldict['Typology'].append(typology)
                sqldict["ListProcessDate"].append(processdate)
                sqldict["RegulationType"].append('Regulated')
                sqldict["RegCtry"].append("MY")
                sqldict["RegCode"].append("CBMAL")
                sqldict["ListCode"].append(reg.split()[-1])
                for key in sqldict:
                    if len(sqldict['Name'])>len(sqldict[key]):
                        sqldict[key].append('')
        else:#all data in the unique list's URL
            ent_list = soup.find('div', {'class','article-content page-content'}).find('ol')
            if ent_list is not None:#case of ordered list (ol) with or contact information
                entities = ent_list.find_all('li')
                entities = [ent_.text.strip() for ent_ in entities]
                for entity in entities:
                    if '***' not in entity:
                        sqldict['Name'].append(entity)
                    else:#name and address/zip
                        ent_split = [ele.strip() for ele in entity.split('***') if len(ele.strip())>1]
                        sqldict['Name'].append(ent_split[0])
                        if len(ent_split)>2 and ent_split[-2].split()[0].isdigit():
                            zip_city = ent_split[-2].split(' ', 1)
                            sqldict['Zip'].append(zip_city[0])
                            if len(zip_city)>1:#there is one case where there is only zip and no City
                                sqldict['City'].append(zip_city[1])
                            sqldict['Address_1'].append(', '.join(ent_split[:-2]))
                        else:
                            sqldict['Address_1'].append(', '.join(ent_split))
                    sqldict['Typology'].append(typology)
                    sqldict["ListProcessDate"].append(processdate)
                    sqldict["RegulationType"].append('Regulated')
                    sqldict["RegCtry"].append("MY")
                    sqldict["RegCode"].append("CBMAL")
                    sqldict["ListCode"].append(reg.split()[-1])
                    for key in sqldict:
                        if len(sqldict['Name'])>len(sqldict[key]):
                            sqldict[key].append('')
            else:#case of table (with contact and without anchor)
                trs = soup.find('div', {'class','article-content page-content'}).find('table').find_all('tr')
                for rang in range(len(trs)):
                    if trs[rang].find('th') is not None:
                        continue
                    elif 'NO.' in trs[rang].find('td').text.upper():
                        continue
                    else:
                        trs = trs[rang:]
                        break
                for tr in trs:
                    tds = tr.find_all('td')
                    sqldict['Name'].append(tds[1].text.strip())
                    if '***' in tds[-1].text:
                        contact_split = [td.strip() for td in tds[-1].text.split('***') if len(td.strip())>1]
                        for cont in contact_split:
                            if 'HQ Address:' in cont and len(sqldict['Name'])>len(sqldict['Address_1']):
                                sqldict['Address_1'].append(cont.split(':',1)[1].strip())
                            elif 'Telephone:' in cont and len(sqldict['Name'])>len(sqldict['Phone']):
                                sqldict['Phone'].append(cont.split(':',1)[1].strip())
                            elif 'Website:' in cont and len(sqldict['Name'])>len(sqldict['Website']):
                                sqldict['Website'].append(cont.split(':',1)[1].strip())
                            elif 'Facsimile:' in cont and len(sqldict['Name'])>len(sqldict['Fax']):
                                sqldict['Fax'].append(cont.split(':',1)[1].strip())
                            elif 'E-mail:' in cont and len(sqldict['Name'])>len(sqldict['Email']):
                                sqldict['Email'].append(cont.split(':',1)[1].strip())
                    sqldict['Typology'].append(typology)
                    sqldict["ListProcessDate"].append(processdate)
                    sqldict["RegulationType"].append('Regulated')
                    sqldict["RegCtry"].append("MY")
                    sqldict["RegCode"].append("CBMAL")
                    sqldict["ListCode"].append(reg.split()[-1])
                    for key in sqldict:
                        if len(sqldict['Name'])>len(sqldict[key]):
                            sqldict[key].append('')
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()
    
    
    